In [1]:
UNI_RANDOM_SEED = 2024

import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(UNI_RANDOM_SEED) 
torch.manual_seed(UNI_RANDOM_SEED)

torch.cuda.manual_seed(UNI_RANDOM_SEED)
torch.cuda.manual_seed_all(UNI_RANDOM_SEED)

import pdb
from pathlib import Path

try:
    import open3d
    from visual_utils import open3d_vis_utils as V
    OPEN3D_FLAG = True
except:
    import mayavi.mlab as mlab
    from visual_utils import visualize_utils as V
    OPEN3D_FLAG = False

from pcdet.datasets.kitti.kitti_dataset import create_kitti_infos
from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.datasets import KittiDataset, build_dataloader
from pcdet.models import build_network, load_data_to_gpu
from pcdet.utils import common_utils

from eval_utils import eval_utils


EVAL_OUTPUT_DIR = "./eval_output/"
CFG_FILE = "./cfgs/kitti_models/pointrcnn.yaml"
DATA_CONFIG_FILE = "./cfgs/dataset_configs/kitti_dataset.yaml"
DATA_PATH = "/home/ksas/Public/datasets/KITTI"
CKPT_PATH = "/home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth"

BATCH_SIZE = 1
WORKERS = 4
DIST_TEST = False

cfg_from_yaml_file(CFG_FILE, cfg)

# BATCH_SIZE = cfg.OPTIMIZATION.BATCH_SIZE_PER_GPU
logger = common_utils.create_logger()
logger.info('-----------------Gradient Fetching Test-------------------------')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


/home/ksas/chw_space/OpenPCDet_Developing/pcdet/models/detectors/__init__.py:20: UserWarning: You are using a variant OpenPCDet modified by uzuki-dev, NOT AN ORIGINAL VERSION!
  warnings.warn("You are using a variant OpenPCDet modified by uzuki-dev, NOT AN ORIGINAL VERSION!")
2024-01-05 21:01:08,923   INFO  -----------------Gradient Fetching Test-------------------------


In [2]:
test_set, test_loader, sampler = build_dataloader(
        dataset_cfg=cfg.DATA_CONFIG,
        class_names=cfg.CLASS_NAMES,
        batch_size=BATCH_SIZE,
        dist=DIST_TEST, workers=WORKERS, logger=logger, training=False
    )
logger.info(f'Class names of samples: \t{test_set.class_names}')

2024-01-05 21:01:08,941   INFO  Loading KITTI dataset
2024-01-05 21:01:09,030   INFO  Total samples for KITTI dataset: 3769
2024-01-05 21:01:09,031   INFO  Class names of samples: 	['Car', 'Pedestrian', 'Cyclist']


In [3]:
model = build_network(model_cfg=cfg.MODEL, num_class=len(cfg.CLASS_NAMES), dataset=test_set)
model.load_params_from_file(filename=CKPT_PATH, logger=logger, to_cpu=True)
model.cuda()
model.eval()

for idx, module in enumerate(model.module_list):
    logger.info(f'Module names of model \t({idx}): \t{module._get_name()}')
    
backbone_network = model.module_list[0]
point_headbox = model.module_list[1]
pointrcnn_head = model.module_list[2]

2024-01-05 21:01:09,821   INFO  ==> Loading parameters from checkpoint /home/ksas/Public/model_zoo/pcdet/pointrcnn_7870.pth to CPU
2024-01-05 21:01:09,843   INFO  ==> Done (loaded 309/309)
2024-01-05 21:01:09,856   INFO  Module names of model 	(0): 	PointNet2MSG
2024-01-05 21:01:09,856   INFO  Module names of model 	(1): 	PointHeadBox
2024-01-05 21:01:09,857   INFO  Module names of model 	(2): 	PointRCNNHead


In [5]:

def pseudo_train_test():
    for i, batch_dict in enumerate(test_loader):
        load_data_to_gpu(batch_dict)
        logger.info(f"keys of batch dict: \t{batch_dict.keys()}")
        
        model.eval()
        model.pseudo_train()
        model.zero_grad()
        pred_dicts, _ = model(batch_dict)
        # pred_dicts, _, _ = model(batch_dict)

        
        loss, tb_dict, disp_dict = model.get_training_loss()
        logger.info(f"total loss: \t{loss}")
        
        loss_dict = {}
       
        point_headbox_cls_loss, cls_loss_dict = point_headbox.get_cls_layer_loss()
        point_headbox_box_loss, box_loss_dict = point_headbox.get_box_layer_loss()
        loss_dict.update(cls_loss_dict)
        loss_dict.update(box_loss_dict)
        
        rcnn_cls_loss, cls_loss_dict = pointrcnn_head.get_box_cls_layer_loss()
        rcnn_reg_loss, reg_loss_dict = pointrcnn_head.get_box_reg_layer_loss()
        loss_dict.update(cls_loss_dict)
        loss_dict.update(reg_loss_dict)
        
        logger.info(f"loss dict: \t{loss_dict}")
        
        pdb.set_trace()
        
pseudo_train_test()

2024-01-05 21:02:48,130   INFO  keys of batch dict: 	dict_keys(['frame_id', 'calib', 'gt_boxes', 'points', 'lidar_aug_matrix', 'use_lead_xyz', 'image_shape', 'batch_size'])
2024-01-05 21:02:48,207   INFO  total loss: 	1.5228084325790405
2024-01-05 21:02:48,214   INFO  loss dict: 	{'point_loss_cls': 0.3448042571544647, 'point_pos_num': 27.0, 'point_loss_box': 0.7239388227462769, 'rcnn_loss_cls': 0.07312199473381042, 'rcnn_loss_reg': 0.3072994649410248, 'rcnn_loss_corner': 0.07364382594823837}


> /tmp/ipykernel_2780001/3186585947.py(2)pseudo_train_test()
      1 def pseudo_train_test():
----> 2     for i, batch_dict in enumerate(test_loader):
      3         load_data_to_gpu(batch_dict)
      4         logger.info(f"keys of batch dict: \t{batch_dict.keys()}")
      5 



In [ ]:
# eval_utils.eval_one_epoch(
#         cfg, None, model, test_loader, 0, logger, dist_test=DIST_TEST,
#         result_dir=None # Path(EVAL_OUTPUT_DIR)
#         , infer_time=True
#     )